In [ ]:
from pathlib import Path
import json
import os
import re
import shutil
import subprocess
import sys
import zipfile

WORKSPACE = Path('/kaggle/working')
S3DIS_INPUT_DIR = Path('/kaggle/input/datasets/ayukiseleva/s3dis-meow/Stanford3dDataset_v1.2')
SOFTGROUP_REPO_URL = 'https://github.com/thangvubk/SoftGroup.git'
SOFTGROUP_DIR = WORKSPACE / 'SoftGroup_crop3'
S3DIS_REPO_DIR = SOFTGROUP_DIR / 'dataset' / 's3dis'
S3DIS_REPO_RAW_DIR = S3DIS_REPO_DIR / 'Stanford3dDataset_v1.2'
CROP_PREPROCESSED_DIR = WORKSPACE / 's3dis_crop3_preprocessed'
OUTPUT_ZIP = WORKSPACE / 's3dis-crop3-preprocessed.zip'

SELECTED_AREAS = ['Area_1', 'Area_2', 'Area_3']
KEEP_ROOMS = ['office_1', 'office_2', 'office_3']
FORCE_REBUILD_PREPROCESS = True

PROJECT_REPO = 'github.com/kslannmnd/point_cloud_playground_play.git'
PROJECT_DIR = WORKSPACE / 'point_cloud_playground_play'
GITHUB_TOKEN_SECRET = 'GITHUB_TOKEN'

TEST_AREA = 'Area_3'
TEST_ROOMS = ['office_1', 'office_2', 'office_3']
AREA = TEST_AREA
ROOM = 'office_3'
OBJECT_NAME = 'chair_1'
R3D_FILE = None  # Example: Path('/kaggle/input/my-r3d/scene.r3d')

RUN_PROJECT_ENV_SETUP = True
RUN_PRETRAINED_DEMO = True
RUN_TRAINING = True
RUN_TRAINED_DEMO = True
RUN_R3D_INFERENCE = R3D_FILE is not None

PRETRAINED_CHECKPOINT = 'outputs/checkpoints/softgroup_s3dis_pretrained.pth'
TRAINED_CHECKPOINT = 'outputs/checkpoints/crop_area123_template_train_full_softgroup_latest.pth'
CHECKPOINT = PRETRAINED_CHECKPOINT

def run(cmd, cwd=None, env=None, check=True):
    printable = ' '.join(map(str, cmd)) if isinstance(cmd, (list, tuple)) else cmd
    print('\n[RUN]', printable, '\n')
    return subprocess.run(cmd, cwd=str(cwd) if cwd else None, env=env, shell=isinstance(cmd, str), check=check)

def safe_remove(path: Path):
    if not path.exists() and not path.is_symlink():
        return
    if path.is_symlink() or path.is_file():
        path.unlink()
    else:
        shutil.rmtree(path)

assert S3DIS_INPUT_DIR.exists(), f'Missing S3DIS input: {S3DIS_INPUT_DIR}'

## 1. Clone Official SoftGroup and Install Light Preprocess Dependencies

In [ ]:
run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'pip', 'wheel', 'setuptools'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy<2', 'pandas', 'pyyaml', 'tqdm', 'plyfile'])

if not SOFTGROUP_DIR.exists():
    run(['git', 'clone', SOFTGROUP_REPO_URL, str(SOFTGROUP_DIR)])
else:
    print('SoftGroup already exists:', SOFTGROUP_DIR)

S3DIS_REPO_DIR.mkdir(parents=True, exist_ok=True)

## 2. Crop Raw S3DIS: Three Rooms from Area_1, Area_2, Area_3

In [ ]:
def copy_crop(src_root: Path, dst_root: Path, selected_areas, keep_rooms):
    safe_remove(dst_root)
    dst_root.mkdir(parents=True, exist_ok=True)
    copied = []
    missing = []
    selected_areas = set(selected_areas)
    for area_dir in sorted(p for p in src_root.iterdir() if p.is_dir() and p.name in selected_areas):
        dst_area = dst_root / area_dir.name
        dst_area.mkdir(parents=True, exist_ok=True)
        for room in keep_rooms:
            src_room = area_dir / room
            if src_room.is_dir():
                shutil.copytree(src_room, dst_area / room)
                copied.append(f'{area_dir.name}/{room}')
            else:
                missing.append(f'{area_dir.name}/{room}')
    print('Copied rooms:', len(copied))
    print('Missing rooms:', len(missing))
    for item in missing[:30]:
        print('MISS', item)
    return copied, missing

copied, missing = copy_crop(S3DIS_INPUT_DIR, S3DIS_REPO_RAW_DIR, SELECTED_AREAS, KEEP_ROOMS)
assert copied, 'No rooms were copied. Check KEEP_ROOMS and input dataset layout.'
print('Cropped raw dataset:', S3DIS_REPO_RAW_DIR)

## 3. Patch SoftGroup S3DIS Parser for Kaggle Input Robustness

In [ ]:
prepare_inst_path = S3DIS_REPO_DIR / 'prepare_data_inst.py'
assert prepare_inst_path.exists(), f'Missing SoftGroup S3DIS parser: {prepare_inst_path}'
src = prepare_inst_path.read_text(encoding='utf-8')
patched = src
room_replacement = "room_ver = pd.read_csv(raw_path, sep=r'\\s+', header=None, engine='python').apply(pd.to_numeric, errors='coerce').dropna(axis=0, how='any').values"
obj_replacement = "obj_ver = pd.read_csv(single_object, sep=r'\\s+', header=None, engine='python').apply(pd.to_numeric, errors='coerce').dropna(axis=0, how='any').values"
patched = re.sub(
    r"room_ver\s*=\s*pd\.read_csv\(\s*raw_path\s*,\s*sep=['\"] ['\"]\s*,\s*header=None\s*\)\.values",
    lambda _: room_replacement,
    patched,
)
patched = re.sub(
    r"obj_ver\s*=\s*pd\.read_csv\(\s*single_object\s*,\s*sep=['\"] ['\"]\s*,\s*header=None\s*\)\.values",
    lambda _: obj_replacement,
    patched,
)
patched = patched.replace(
    "rgb = np.ascontiguousarray(room_ver[:, 3:6], dtype='uint8')",
    "rgb = np.clip(np.rint(np.ascontiguousarray(room_ver[:, 3:6], dtype='float32')), 0, 255).astype('uint8')",
)
patched = patched.replace('range(1, 7)', 'range(1, 4)')
patched = patched.replace('range(1,7)', 'range(1,4)')
if patched != src:
    prepare_inst_path.write_text(patched, encoding='utf-8')
    print('Patched:', prepare_inst_path)
else:
    print('Parser/area patch already applied or not needed.')

for extra_script in [S3DIS_REPO_DIR / 'prepare_data.py']:
    if not extra_script.exists():
        continue
    extra_src = extra_script.read_text(encoding='utf-8')
    extra_patched = extra_src.replace('range(1, 7)', 'range(1, 4)').replace('range(1,7)', 'range(1,4)')
    if extra_patched != extra_src:
        extra_script.write_text(extra_patched, encoding='utf-8')
        print('Patched area range:', extra_script)

print('Existing raw areas:', sorted(p.name for p in S3DIS_REPO_RAW_DIR.iterdir() if p.is_dir()))

## 4. Run SoftGroup Preprocessing on Cropped Dataset

In [ ]:
if FORCE_REBUILD_PREPROCESS:
    for name in ['preprocess', 'preprocess_sample', 'val_gt']:
        safe_remove(S3DIS_REPO_DIR / name)

run(['bash', 'prepare_data.sh'], cwd=S3DIS_REPO_DIR)

processed_files = sorted((S3DIS_REPO_DIR / 'preprocess').glob('*_inst_nostuff.pth'))
print('Processed room files:', len(processed_files))
assert processed_files, 'prepare_data.sh finished but preprocess/*.pth is empty'

## 5. Materialize Cropped Preprocessed Dataset and Zip Artifact

In [ ]:
safe_remove(CROP_PREPROCESSED_DIR)
CROP_PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)
for name in ['preprocess', 'preprocess_sample', 'val_gt']:
    src = S3DIS_REPO_DIR / name
    if src.exists():
        shutil.copytree(src, CROP_PREPROCESSED_DIR / name)

metadata = {
    'source': str(S3DIS_INPUT_DIR),
    'crop_keep_rooms': KEEP_ROOMS,
    'copied_rooms': copied,
    'missing_rooms': missing,
    'processed_files': len(processed_files),
    'contains': ['preprocess', 'preprocess_sample', 'val_gt'],
}
(CROP_PREPROCESSED_DIR / 'preprocess_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')

safe_remove(OUTPUT_ZIP)
with zipfile.ZipFile(OUTPUT_ZIP, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=4) as zf:
    for path in CROP_PREPROCESSED_DIR.rglob('*'):
        if path.is_file():
            zf.write(path, arcname=str(path.relative_to(CROP_PREPROCESSED_DIR)))

print('Cropped preprocessed dataset:', CROP_PREPROCESSED_DIR)
print('Zip artifact:', OUTPUT_ZIP)
print('Zip size GB:', OUTPUT_ZIP.stat().st_size / 1024**3)

## 6. Clone This Private Project Repository

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret(GITHUB_TOKEN_SECRET)
except Exception as exc:
    raise RuntimeError(f'Create Kaggle secret {GITHUB_TOKEN_SECRET!r} with private repo access first.') from exc

clone_url = f'https://{token}@{PROJECT_REPO}'
if PROJECT_DIR.exists():
    print('Project already exists:', PROJECT_DIR)
else:
    run(['git', 'clone', clone_url, str(PROJECT_DIR)])
run(['git', 'remote', 'set-url', 'origin', f'https://{PROJECT_REPO}'], cwd=PROJECT_DIR, check=False)
print('Project dir:', PROJECT_DIR)

## 7. Install Project and Link Cropped Preprocessed Dataset

In [ ]:
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], cwd=PROJECT_DIR)

COMMON = [
    'env=kaggle',
    'dataset=s3dis_preprocessed',
    'inference=s3dis',
    f'dataset.preprocessed.source_dir={CROP_PREPROCESSED_DIR}',
    'dataset.fold.preferred_test_area=Area_3',
]

if RUN_PROJECT_ENV_SETUP:
    run([sys.executable, 'scripts/install_dependencies_and_build_softgroup.py', *COMMON], cwd=PROJECT_DIR)
    run([sys.executable, 'scripts/prepare_s3dis.py', *COMMON], cwd=PROJECT_DIR)
    run([sys.executable, 'scripts/download_softgroup_checkpoints.py', *COMMON], cwd=PROJECT_DIR)
    run([sys.executable, 'scripts/patch_softgroup_for_kaggle.py', *COMMON], cwd=PROJECT_DIR)
    run([sys.executable, 'scripts/generate_softgroup_configs.py', *COMMON], cwd=PROJECT_DIR)
else:
    print('RUN_PROJECT_ENV_SETUP=False')

## 8. Show Available Cropped Rooms

In [ ]:
run([sys.executable, 'scripts/show_s3dis_rooms.py', *COMMON], cwd=PROJECT_DIR)

## 9. Pretrained Checkpoint: Inference, Metrics, and BBox HTML for Area_3

In [ ]:
if RUN_PRETRAINED_DEMO:
    CHECKPOINT = PRETRAINED_CHECKPOINT
    run([sys.executable, 'scripts/precompute_s3dis_inference.py', TEST_AREA, *COMMON, f'inference.checkpoint={CHECKPOINT}'], cwd=PROJECT_DIR)
    run([
        sys.executable, 'scripts/compute_s3dis_metrics.py', *COMMON,
        f'metrics.checkpoint={CHECKPOINT}',
        "metrics.rooms=[Area_3/office_1,Area_3/office_2,Area_3/office_3]",
    ], cwd=PROJECT_DIR)
    run([sys.executable, 'scripts/show_room_bbox.py', TEST_AREA, 'office_3', *COMMON, f'inference.checkpoint={CHECKPOINT}'], cwd=PROJECT_DIR)
else:
    print('RUN_PRETRAINED_DEMO=False')

## 10. Display Latest Pretrained HTML

In [ ]:
from IPython.display import HTML, display
html_candidates = []
for root in [PROJECT_DIR / 'outputs' / 'visualizations' / 's3dis', PROJECT_DIR / 'outputs' / 'inference']:
    if root.exists():
        html_candidates.extend(sorted(root.rglob('*.html')))
print('HTML files:', len(html_candidates))
if html_candidates:
    latest_html = html_candidates[-1]
    print('Displaying:', latest_html)
    display(HTML(latest_html.read_text(encoding='utf-8')))

## 11. Train S3DIS Experiment on Area_1/Area_2, Test Area_3

In [ ]:
if RUN_TRAINING:
    run([
        sys.executable, 'scripts/train.py', *COMMON,
        'training=template_train',
        'training.experiment_name=crop_area123_template_train',
        'training.epochs=2',
        'training.lr=0.001',
        'training.train_batch_size=1',
        'training.num_workers=2',
    ], cwd=PROJECT_DIR)
else:
    print('RUN_TRAINING=False')

## 12. Trained Checkpoint: Inference, Metrics, and BBox HTML for Area_3

In [ ]:
if RUN_TRAINED_DEMO:
    CHECKPOINT = TRAINED_CHECKPOINT if (PROJECT_DIR / TRAINED_CHECKPOINT).exists() else PRETRAINED_CHECKPOINT
    print('Using checkpoint:', CHECKPOINT)
    run([sys.executable, 'scripts/precompute_s3dis_inference.py', TEST_AREA, *COMMON, f'inference.checkpoint={CHECKPOINT}', 'visualization.force_run=true'], cwd=PROJECT_DIR)
    run([
        sys.executable, 'scripts/compute_s3dis_metrics.py', *COMMON,
        f'metrics.checkpoint={CHECKPOINT}',
        "metrics.rooms=[Area_3/office_1,Area_3/office_2,Area_3/office_3]",
        'metrics.force_run=true',
    ], cwd=PROJECT_DIR)
    run([sys.executable, 'scripts/show_room_bbox.py', TEST_AREA, 'office_3', *COMMON, f'inference.checkpoint={CHECKPOINT}', 'visualization.force_run=true'], cwd=PROJECT_DIR)
else:
    print('RUN_TRAINED_DEMO=False')

## 13. Display Latest Saved S3DIS HTML

In [ ]:
from IPython.display import HTML, display
html_candidates = []
for root in [PROJECT_DIR / 'outputs' / 'visualizations' / 's3dis', PROJECT_DIR / 'outputs' / 'inference']:
    if root.exists():
        html_candidates.extend(sorted(root.rglob('*.html')))
print('HTML files:', len(html_candidates))
if html_candidates:
    latest_html = html_candidates[-1]
    print('Displaying:', latest_html)
    display(HTML(latest_html.read_text(encoding='utf-8')))

## 14. Optional R3D Inference with the Same Checkpoint

In [ ]:
if RUN_R3D_INFERENCE:
    run([
        sys.executable, 'run_r3d_inference.py', str(R3D_FILE),
        'env=kaggle', 'dataset=r3d', 'inference=r3d',
        f'inference.checkpoint={CHECKPOINT}',
    ], cwd=PROJECT_DIR)
else:
    print('RUN_R3D_INFERENCE=False. Set R3D_FILE to a Kaggle input .r3d path to run.')

## 15. Display Latest R3D HTML

In [ ]:
r3d_root = PROJECT_DIR / 'outputs' / 'r3d_inference'
if r3d_root.exists():
    html_files = sorted(r3d_root.rglob('point_cloud.html'))
    print('R3D HTML files:', len(html_files))
    if html_files:
        display(HTML(html_files[-1].read_text(encoding='utf-8')))
else:
    print('No R3D outputs yet.')